In [1]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [2]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM mbmeg10", con=connection2)

connection2.close()

In [3]:
base2

,motegrcod,motegrdes,motegrresatenrel
0,01,ALTA MEDICA,02
1,02,RETIRO VOLUNTARIO,31
2,03,A CONSULTA EXTERNA,02
3,04,REFERENCIA,04
4,05,TRANSFERENCIA C.A. NO ESSALUD,22
5,06,TRANSFERENCIA HOSPITALIZACION,06
6,07,TRANSFERENCIA CENT.QUIRURGICO\n,07
7,12,FALLECIMIENTO,99
8,11,FUGA,32
9,08,TRANSFERENCIA UCI,15


In [4]:
a=base2[base2.duplicated(subset=["motegrcod"])]
a

,motegrcod,motegrdes,motegrresatenrel


In [5]:
base2.columns

Index(['motegrcod', 'motegrdes', 'motegrresatenrel'], dtype='object')

In [6]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



base2.to_sql(name='tmp_mbmeg10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL mbmeg10 FALSO CON LO OBTENIDO DEL ORACLE

query_count_before = "SELECT COUNT(*) FROM mbmeg10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla mbmeg10 antes de la inserción: {cant_antes}")



query="""
ALTER TABLE tmp_mbmeg10 
ALTER COLUMN motegrcod TYPE CHARACTER (2),
ALTER COLUMN motegrdes TYPE CHARACTER (30),
ALTER COLUMN motegrresatenrel TYPE CHARACTER (2);


UPDATE mbmeg10 
SET 

motegrdes= t.motegrdes,
motegrresatenrel= t.motegrresatenrel


FROM tmp_mbmeg10 t 
WHERE mbmeg10.motegrcod = t.motegrcod and mbmeg10.motegrcod IS NOT NULL and t.motegrcod IS NOT NULL ;

INSERT INTO mbmeg10 
(motegrcod, motegrdes,motegrresatenrel) 

SELECT 
motegrcod, motegrdes,motegrresatenrel

FROM tmp_mbmeg10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM mbmeg10 
    WHERE mbmeg10.motegrcod = tmp_mbmeg10.motegrcod and mbmeg10.motegrcod IS NOT NULL and tmp_mbmeg10.motegrcod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_mbmeg10;
DELETE FROM mbmeg10 WHERE motegrcod ='';
SELECT COUNT(*) FROM mbmeg10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbmeg10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

base2 = pd.read_sql_query(f"SELECT * FROM mbmeg10", con=connection3)

connection3.close()


Cantidad de filas en la tabla mbmeg10 antes de la inserción: 13
Cantidad de filas en la tabla mbmeg10 después de la inserción: 13
La cantidad de filas insertadas fue de: 0


In [7]:
base2.columns

Index(['motegrcod', 'motegrdes', 'motegrresatenrel'], dtype='object')

In [8]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_mbmeg10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query_count_before = "SELECT COUNT(*) FROM dim_motegr"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_motegr antes de la inserción: {cant_antes}")


query="""
ALTER TABLE tmp_mbmeg10 
ALTER COLUMN motegrcod TYPE CHARACTER (2),
ALTER COLUMN motegrdes TYPE CHARACTER (30),
ALTER COLUMN motegrresatenrel TYPE CHARACTER (2);



INSERT INTO dim_motegr (cod_mot, des_mot,mot_egr) 
SELECT motegrcod, motegrdes,motegrresatenrel

FROM tmp_mbmeg10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_motegr 
    WHERE dim_motegr.cod_mot = tmp_mbmeg10.motegrcod
);

DROP TABLE tmp_mbmeg10;

SELECT COUNT(*) FROM dim_motegr;
"""

c1 = text(query)
cursor = connection4.execute(c1)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla dim_motegr después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection4.close()

Cantidad de filas en la tabla dim_motegr antes de la inserción: 13
Cantidad de filas en la tabla dim_motegr después de la inserción: 13
La cantidad de filas insertadas fue de: 0


In [9]:
connection3.close()